# 🏎️ Validation - Alpha Comparison


### Outputs
| File | Description |
|------|-------------|
| `alpha_<track>.csv` × 10 | `point_index, alpha_calculated, alpha_predicted, error, abs_error_cm` |
| `racingline_<track>.png` × 10 | Full track layout — calculated (blue) vs predicted (red→green by error) |
| `alpha_comparison_outputs.zip` | All of the above bundled |


## Step 0 — Imports

In [ ]:
import os, io, re, json, math, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
warnings.filterwarnings('ignore')
print("Imports OK")


## Step 1 — Upload Files

In [ ]:
from google.colab import files as _files
print("Upload predictions.json + 10 ground truth CSVs (11 files total):")
uploaded = _files.upload()


## Step 2 — Parse Uploads

In [ ]:
import re

def normalise_name(fname):
    name = re.sub(r'^\d+_', '', fname)
    name = re.sub(r'\.(csv|CSV|json)$', '', name, flags=re.IGNORECASE)
    name = name.replace('_', ' ')
    name = re.sub(r'\s+ground\s+truth\s*$', '', name, flags=re.IGNORECASE).strip()
    return name

def simplify(s):
    return re.sub(r'[^a-z0-9]', '', s.lower())

def is_truncated(name):
    return '~' in name

def lcs_len(a, b):
    a, b = simplify(a), simplify(b)
    best = 0
    for i in range(len(a)):
        for j in range(len(b)):
            l = 0
            while i+l < len(a) and j+l < len(b) and a[i+l] == b[j+l]:
                l += 1
            best = max(best, l)
    return best

def match_tracks(pred_names, csv_names):
    remaining = list(pred_names)
    non_trunc = [c for c in csv_names if not is_truncated(c)]
    trunc     = sorted([c for c in csv_names if is_truncated(c)])
    matched   = {}
    for ct in non_trunc:
        if not remaining:
            break
        best_pt = max(remaining, key=lambda pt: lcs_len(pt, ct))
        if lcs_len(best_pt, ct) >= 4:
            matched[best_pt] = ct
            remaining.remove(best_pt)
    for pt, ct in zip(sorted(remaining), trunc):
        matched[pt] = ct
    return matched

predictions_raw = None
track_data      = {}

for fname, content in uploaded.items():
    if 'prediction' in fname.lower() or fname.lower().endswith('.json'):
        predictions_raw = json.loads(content.decode('utf-8'))
        print(f"  ✓ predictions.json  ({len(predictions_raw)} tracks)")
    else:
        df   = pd.read_csv(io.BytesIO(content))
        name = normalise_name(fname)
        track_data[name] = df
        print(f"  ✓ {name:50s}  {len(df)} rows")

if predictions_raw is None:
    raise ValueError("predictions.json not found — please upload it.")
print(f"\nCSVs loaded: {len(track_data)}")

pred_names = list(predictions_raw.keys())
csv_names  = list(track_data.keys())
matched    = match_tracks(pred_names, csv_names)

print("\n=== Matching results ===")
for pt, ct in matched.items():
    flag = "  ~TRUNCATED" if is_truncated(ct) else ""
    print(f"  PRED: '{pt}' -> CSV: '{ct}'{flag}")

unmatched = [p for p in pred_names if p not in matched]
if unmatched:
    print(f"\n WARNING Unmatched predictions: {unmatched}")
print(f"Matched {len(matched)}/10 tracks.")


## Step 3 — Constants & Wall Helper

In [ ]:
TRACK_WIDTH = 9.0  # metres

def get_walls(df):
    REF_LAT = df['t_lat'].mean()
    MLAT = 111320.0
    MLON = 111320.0 * math.cos(math.radians(REF_LAT))
    cx = (df['t_lon'].values - df['t_lon'].mean()) * MLON
    cy = (df['t_lat'].values - df['t_lat'].mean()) * MLAT
    dx = np.gradient(cx); dy = np.gradient(cy)
    norm = np.sqrt(dx**2 + dy**2)
    norm = np.where(norm < 1e-9, 1e-9, norm)
    px = -dy / norm; py = dx / norm
    dist_inner = df['dist_inner'].values if 'dist_inner' in df.columns else np.full(len(df), TRACK_WIDTH/2)
    dist_outer = df['dist_outer'].values if 'dist_outer' in df.columns else np.full(len(df), TRACK_WIDTH/2)
    inner_x = cx - dist_inner * px;  inner_y = cy - dist_inner * py
    outer_x = cx + dist_outer * px;  outer_y = cy + dist_outer * py
    return cx, cy, px, py, dist_inner, dist_outer, inner_x, inner_y, outer_x, outer_y

def alpha_to_xy(cx, cy, px, py, dist_inner, dist_outer, alpha):
    tw     = dist_inner + dist_outer
    offset = -dist_inner + alpha * tw
    return cx + offset * px, cy + offset * py

print("Helpers ready.")


## Step 4 — Generate Alpha CSVs

In [ ]:
os.makedirs('alpha_csv', exist_ok=True)

print("Saving alpha comparison CSVs...")
for pred_name, csv_name in matched.items():
    df     = track_data[csv_name]
    y_true = np.array(predictions_raw[pred_name]['y_true'])
    y_pred = np.array(predictions_raw[pred_name]['y_pred'])
    error  = y_pred - y_true
    err_cm = np.abs(error) * TRACK_WIDTH * 100

    out = pd.DataFrame({
        'point_index':      np.arange(len(y_true)),
        'alpha_calculated': np.round(y_true, 6),
        'alpha_predicted':  np.round(y_pred, 6),
        'error':            np.round(error,  6),
        'abs_error_cm':     np.round(err_cm, 3),
    })

    safe = re.sub(r'[^a-zA-Z0-9]', '_', pred_name).strip('_')
    path = f'alpha_csv/alpha_{safe}.csv'
    out.to_csv(path, index=False)
    print(f"  ✓ {pred_name:45s}  MAE={err_cm.mean():.1f} cm  →  alpha_{safe}.csv")

print("\nAll 10 CSVs saved.")


## Step 4b — Accuracy Metrics Summary

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Per-track metrics ──────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 100)
print(f"{'ALPHA PREDICTION ACCURACY — v8_fixed (RacingLineTransformer, 60.9 cm mean MAE)'}")
print("=" * 100)
print(f"{'Track':<45} | {'MAE (cm)':>9} | {'RMSE (cm)':>9} | {'R²':>7} | {'Corr':>7} | {'Correct Side':>13} | {'Max Err (cm)':>13}")
print("-" * 100)

all_metrics = []

for pred_name, csv_name in matched.items():
    y_true = np.array(predictions_raw[pred_name]['y_true'])
    y_pred = np.array(predictions_raw[pred_name]['y_pred'])

    mae_cm   = mean_absolute_error(y_true, y_pred) * TRACK_WIDTH * 100
    rmse_cm  = np.sqrt(mean_squared_error(y_true, y_pred)) * TRACK_WIDTH * 100
    r2       = r2_score(y_true, y_pred)
    corr     = np.corrcoef(y_true, y_pred)[0, 1]
    correct  = np.mean((y_true > 0.5) == (y_pred > 0.5)) * 100
    max_err  = np.max(np.abs(y_true - y_pred)) * TRACK_WIDTH * 100
    bias_cm  = np.mean(y_pred - y_true) * TRACK_WIDTH * 100

    all_metrics.append({
        'track':       pred_name,
        'mae_cm':      mae_cm,
        'rmse_cm':     rmse_cm,
        'r2':          r2,
        'corr':        corr,
        'correct_pct': correct,
        'max_err_cm':  max_err,
        'bias_cm':     bias_cm,
    })

    r2_str   = f"{r2:+.3f}"
    corr_str = f"{corr:.3f}"
    print(f"  {pred_name:<43} | {mae_cm:>9.1f} | {rmse_cm:>9.1f} | {r2_str:>7} | {corr_str:>7} | {correct:>12.1f}% | {max_err:>12.1f}")

# ── Summary statistics ─────────────────────────────────────────────────────────
maes   = [m['mae_cm']      for m in all_metrics]
rmses  = [m['rmse_cm']     for m in all_metrics]
r2s    = [m['r2']          for m in all_metrics]
corrs  = [m['corr']        for m in all_metrics]
sides  = [m['correct_pct'] for m in all_metrics]
maxes  = [m['max_err_cm']  for m in all_metrics]

print("-" * 100)
print(f"  {'MEAN':<43} | {np.mean(maes):>9.1f} | {np.mean(rmses):>9.1f} | {np.mean(r2s):>+7.3f} | {np.mean(corrs):>7.3f} | {np.mean(sides):>12.1f}% | {np.mean(maxes):>12.1f}")
print(f"  {'STD':<43} | {np.std(maes):>9.1f} | {np.std(rmses):>9.1f} | {np.std(r2s):>+7.3f} | {np.std(corrs):>7.3f} | {np.std(sides):>12.1f}% |")
print(f"  {'BEST':<43} | {min(maes):>9.1f} | {min(rmses):>9.1f} | {max(r2s):>+7.3f} | {max(corrs):>7.3f} | {max(sides):>12.1f}% | {min(maxes):>12.1f}")
print(f"  {'WORST':<43} | {max(maes):>9.1f} | {max(rmses):>9.1f} | {min(r2s):>+7.3f} | {min(corrs):>7.3f} | {min(sides):>12.1f}% | {max(maxes):>12.1f}")
print("=" * 100)
print()

# ── Interpretation guide ────────────────────────────────────────────────────────
n_good   = sum(1 for m in all_metrics if m['mae_cm'] < 50)
n_decent = sum(1 for m in all_metrics if 50 <= m['mae_cm'] < 100)
n_poor   = sum(1 for m in all_metrics if m['mae_cm'] >= 100)

print("METRIC DEFINITIONS")
print(f"  MAE (cm)     — Mean absolute lateral error in centimetres on a {TRACK_WIDTH:.0f}m-wide track")
print(f"  RMSE (cm)    — Root mean square error; penalises large spikes more than MAE")
print(f"  R²           — Coefficient of determination; 1.0 = perfect, 0.0 = predicts mean, <0 = worse than mean")
print(f"  Corr         — Pearson correlation; how well the shape of the prediction matches ground truth")
print(f"  Correct Side — % of track points where the model correctly predicts inner (α<0.5) vs outer (α>0.5)")
print(f"  Max Err (cm) — Worst single-point error; shows peak lateral deviation at tightest corners")
print()
print("TRACK CLASSIFICATION")
print(f"  Excellent (MAE < 50 cm)   : {n_good}/10 tracks")
print(f"  Acceptable (50–100 cm)    : {n_decent}/10 tracks")
print(f"  Poor (MAE > 100 cm)       : {n_poor}/10 tracks")
print()

# ── Comparison table vs naive baselines ────────────────────────────────────────
print("COMPARISON vs NAIVE BASELINES")
print(f"  Baseline 1 — Always predict centreline (α=0.5 everywhere):")
naive_mae = []
for pred_name in matched.keys():
    y_true = np.array(predictions_raw[pred_name]['y_true'])
    naive_mae.append(np.mean(np.abs(y_true - 0.5)) * TRACK_WIDTH * 100)
print(f"    Mean MAE = {np.mean(naive_mae):.1f} cm")
print(f"  Our model : {np.mean(maes):.1f} cm  →  {(np.mean(naive_mae) - np.mean(maes)) / np.mean(naive_mae) * 100:.1f}% improvement over centreline baseline")
print()

print("CONTEXT (on a 9m track)")
print(f"  60 cm error   ≈ width of a car tyre")
print(f"  30 cm error   ≈ half a tyre width — competitive precision")
print(f"  130 cm error  ≈ width of the car itself — functional but imprecise")


## Step 5 — Racing Line Plots

In [ ]:
os.makedirs('alpha_plots', exist_ok=True)

print("Generating racing line plots...")

for pred_name, csv_name in matched.items():
    df     = track_data[csv_name]
    y_true = np.array(predictions_raw[pred_name]['y_true'])
    y_pred = np.array(predictions_raw[pred_name]['y_pred'])
    err_alpha = np.abs(y_pred - y_true)
    mae_cm    = err_alpha.mean() * TRACK_WIDTH * 100

    cx, cy, px, py, dist_inner, dist_outer, inner_x, inner_y, outer_x, outer_y = get_walls(df)
    calc_x, calc_y = alpha_to_xy(cx, cy, px, py, dist_inner, dist_outer, y_true)
    pred_x, pred_y = alpha_to_xy(cx, cy, px, py, dist_inner, dist_outer, y_pred)

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#1a1a2e')

    # Track surface fill
    wall_x = np.concatenate([inner_x, outer_x[::-1]])
    wall_y = np.concatenate([inner_y, outer_y[::-1]])
    ax.fill(wall_x, wall_y, color='#2a2a3e', alpha=0.7, zorder=0)
    ax.plot(inner_x, inner_y, color='#5a5a7a', linewidth=1.5, zorder=1)
    ax.plot(outer_x, outer_y, color='#5a5a7a', linewidth=1.5, zorder=1)

    # Centreline
    ax.plot(cx, cy, color='#888899', linewidth=0.8, linestyle='--', alpha=0.4, zorder=2)

    # Calculated racing line — solid blue
    ax.plot(calc_x, calc_y, color='#4fc3f7', linewidth=2.5, alpha=0.95,
            label='Calculated racing line', zorder=3, solid_capstyle='round')

    # Predicted racing line — coloured by error (green=accurate, red=large error)
    points   = np.array([pred_x, pred_y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    norm_err = plt.Normalize(0, 0.25)
    lc = LineCollection(segments, cmap='RdYlGn_r', norm=norm_err,
                        linewidth=2.5, alpha=0.95, zorder=4, capstyle='round')
    lc.set_array(err_alpha[:-1])
    ax.add_collection(lc)

    # Gap arrows every 25 points (only where gap > 0.3 m)
    for i in range(0, len(calc_x), 25):
        dx_g = pred_x[i] - calc_x[i]
        dy_g = pred_y[i] - calc_y[i]
        if math.sqrt(dx_g**2 + dy_g**2) > 0.3:
            ax.annotate('', xy=(pred_x[i], pred_y[i]), xytext=(calc_x[i], calc_y[i]),
                        arrowprops=dict(arrowstyle='->', color='#ffcc02', lw=0.9, alpha=0.7),
                        zorder=5)

    # Start dot
    ax.plot(calc_x[0], calc_y[0], 'o', color='white', markersize=9, zorder=6, label='Start')

    # Colourbar
    sm = plt.cm.ScalarMappable(cmap='RdYlGn_r', norm=norm_err)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.02)
    cbar.set_label('Prediction error (α units)', color='white', fontsize=9)
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    # Legend
    c_patch = mpatches.Patch(color='#4fc3f7', label='Calculated racing line')
    p_patch = mpatches.Patch(color='#ff6b6b', label='Predicted line (red=large error, green=small)')
    w_patch = mpatches.Patch(color='#5a5a7a', label='Track walls')
    ax.legend(handles=[c_patch, p_patch, w_patch],
              facecolor='#2a2a3e', labelcolor='white', fontsize=9, loc='upper right')

    ax.set_title(f'{pred_name}\nCalculated vs Predicted Racing Line  |  MAE = {mae_cm:.1f} cm',
                 color='white', fontsize=11, pad=10)
    ax.set_xlabel('Local X (m)', color='white')
    ax.set_ylabel('Local Y (m)', color='white')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#4a4a6a')
    ax.set_aspect('equal')

    safe = re.sub(r'[^a-zA-Z0-9]', '_', pred_name).strip('_')
    path = f'alpha_plots/racingline_{safe}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show(); plt.close()
    print(f"  ✓ {pred_name}")

print("\nAll 10 plots saved.")


## Step 6 — Download All Outputs

In [ ]:
from google.colab import files as _dl
import glob

with zipfile.ZipFile('alpha_comparison_outputs.zip', 'w') as zf:
    for f in glob.glob('alpha_csv/*.csv') + glob.glob('alpha_plots/*.png'):
        zf.write(f)

print("Downloading alpha_comparison_outputs.zip ...")
_dl.download('alpha_comparison_outputs.zip')
print("Done.")
